# Training GoogleNet untuk Klasifikasi Penyakit Tanaman Strawberry

Notebook ini dapat dijalankan di **Google Colab** atau **Kaggle**.

## Setup
- **Colab**: Aktifkan GPU di Runtime > Change runtime type > GPU
- **Kaggle**: Aktifkan GPU di Settings > Accelerator > GPU
- Jika tidak ada GPU, akan otomatis menggunakan CPU

In [1]:
!pip install roboflow -q

from roboflow import Roboflow
import os

rf = Roboflow(api_key="jPzrt0coWYniVrkXMnTN")
project = rf.workspace("pcb-akuxr").project("strawberry-ab4ge")
version = project.version(1)
dataset = version.download("folder")

# Path dataset dari Roboflow (format folder: train/, valid/ atau test/)
if hasattr(dataset, 'location'):
    ROBOFLOW_PATH = dataset.location
elif isinstance(dataset, str):
    ROBOFLOW_PATH = dataset
else:
    ROBOFLOW_PATH = str(dataset)
if not os.path.isabs(ROBOFLOW_PATH):
    ROBOFLOW_PATH = os.path.abspath(ROBOFLOW_PATH)
print(f"Dataset path: {ROBOFLOW_PATH}")
if os.path.exists(ROBOFLOW_PATH):
    print(f"Konten: {os.listdir(ROBOFLOW_PATH)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to strawberry-1 in folder:: 100%|██████████| 19835/19835 [00:03<00:00, 5968.55it/s]


Dataset path: /kaggle/working/strawberry-1
Konten: ['test', 'README.dataset.txt', 'README.roboflow.txt', 'train']


## 1. Install Dependencies

In [2]:
# Uncomment jika di Kaggle (Colab sudah include torch)
!pip install torch torchvision --quiet

## 2. Konfigurasi Dataset dan Path

In [9]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import json

# Deteksi environment (Colab vs Kaggle)
try:
    import google.colab
    IN_COLAB = True
    BASE_PATH = '/content'
except:
    IN_COLAB = False
    BASE_PATH = '/kaggle/working' if os.path.exists('/kaggle') else '.'

# Path dataset - dari Roboflow (cell 1) atau manual
_data_path = globals().get('ROBOFLOW_PATH', '')
DATA_DIR = _data_path if _data_path and os.path.exists(_data_path) else os.path.join(BASE_PATH, 'dataset')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
# Roboflow pakai "valid", beberapa dataset pakai "val"
VAL_DIR = os.path.join(DATA_DIR, 'valid') if os.path.exists(os.path.join(DATA_DIR, 'valid')) else os.path.join(DATA_DIR, 'val')

# ========== WHITELIST LABEL ==========
# Hanya label dalam list ini yang dipakai untuk training. Kosongkan [] = pakai semua label
WHITELIST_LABELS = [
    "Angular Leafspot",
    "Leaf Mildew Powdery",
    "Leaf Spot",
    "normal"
]  # Contoh: ["Leaf_Scorch", "Healthy", "Powdery_Mildew"]

# Output model
OUTPUT_DIR = os.path.join(BASE_PATH, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(OUTPUT_DIR, 'googlenet_model.pth')
CLASSES_PATH = os.path.join(OUTPUT_DIR, 'class_names.json')

# Training config
BATCH_SIZE = 32
EPOCHS = 5
IMG_SIZE = 224
LEARNING_RATE = 0.001

# Deteksi device (GPU/CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Val dir: {VAL_DIR}')
print(f'Whitelist: {WHITELIST_LABELS if WHITELIST_LABELS else "SEMUA LABEL"}')

Using device: cuda
GPU: Tesla T4
Train dir: /kaggle/working/strawberry-1/train
Val dir: /kaggle/working/strawberry-1/val
Whitelist: ['Angular Leafspot', 'Leaf Mildew Powdery', 'Leaf Spot', 'normal']


## 3. Cek Struktur Dataset (opsional)

Jika tidak pakai Roboflow, uncomment di cell bawah untuk upload zip manual. Setelah Roboflow (cell 1), dataset sudah ter-load.

In [10]:
# Cek struktur dataset (pastikan cell 1 & 2 sudah dijalankan)
if os.path.exists(TRAIN_DIR):
    all_classes = sorted(os.listdir(TRAIN_DIR))
    print(f'Semua label di dataset: {all_classes}')
    for c in all_classes:
        p = os.path.join(TRAIN_DIR, c)
        if os.path.isdir(p):
            count = len(os.listdir(p))
            print(f'  - {c}: {count} images')
    print('\nUntuk whitelist: edit WHITELIST_LABELS di cell Konfigurasi. Contoh:')
    print('  WHITELIST_LABELS = ["Leaf_Scorch", "Healthy"]  # hanya 2 kelas')
    print('  WHITELIST_LABELS = []  # semua kelas')
else:
    print(f'TRAIN_DIR not found: {TRAIN_DIR}')
    print('Jalankan cell 1 (Roboflow) terlebih dahulu.')

Semua label di dataset: ['Angular Leafspot', 'Anthracnose Fruit Rot', 'Blight Blossom', 'Fruit Mildew Powdery', 'Gray Mold', 'Leaf Mildew Powdery', 'Leaf Spot', 'normal', 'others']
  - Angular Leafspot: 2629 images
  - Anthracnose Fruit Rot: 606 images
  - Blight Blossom: 1294 images
  - Fruit Mildew Powdery: 816 images
  - Gray Mold: 3005 images
  - Leaf Mildew Powdery: 3370 images
  - Leaf Spot: 3893 images
  - normal: 2010 images
  - others: 1083 images

Untuk whitelist: edit WHITELIST_LABELS di cell Konfigurasi. Contoh:
  WHITELIST_LABELS = ["Leaf_Scorch", "Healthy"]  # hanya 2 kelas
  WHITELIST_LABELS = []  # semua kelas


## 4. Data Loaders

In [11]:
def filter_dataset_by_whitelist(dataset, whitelist):
    """Filter ImageFolder by whitelist. Jika whitelist kosong, return dataset as-is."""
    if not whitelist or len(whitelist) == 0:
        return dataset, dataset.classes
    whitelist_set = {str(w).strip().lower() for w in whitelist}
    old_classes = dataset.classes
    # Indeks kelas yang masuk whitelist (case-insensitive)
    valid_old_indices = [i for i, c in enumerate(old_classes) if c.lower() in whitelist_set]
    if not valid_old_indices:
        raise ValueError(f"Whitelist {whitelist} tidak cocok dengan kelas: {old_classes}")
    old_to_new = {old_i: new_i for new_i, old_i in enumerate(valid_old_indices)}
    new_classes = [old_classes[i] for i in valid_old_indices]
    # Filter samples: (path, old_target) -> (path, new_target)
    filtered_samples = [(p, old_to_new[t]) for p, t in dataset.samples if t in old_to_new]
    dataset.samples = filtered_samples
    dataset.targets = [s[1] for s in filtered_samples]
    dataset.classes = new_classes
    dataset.class_to_idx = {c: i for i, c in enumerate(new_classes)}
    return dataset, new_classes

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if os.path.exists(VAL_DIR):
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
    train_dataset, class_names = filter_dataset_by_whitelist(train_dataset, WHITELIST_LABELS)
    val_dataset, _ = filter_dataset_by_whitelist(val_dataset, WHITELIST_LABELS)
else:
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    train_ds, class_names = filter_dataset_by_whitelist(train_ds, WHITELIST_LABELS)
    val_ds = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)
    val_ds, _ = filter_dataset_by_whitelist(val_ds, WHITELIST_LABELS)
    n = len(train_ds)
    val_size = max(1, n // 5)
    train_dataset = Subset(train_ds, list(range(n - val_size)))
    val_dataset = Subset(val_ds, list(range(n - val_size, n)))

num_classes = len(class_names)

# Simpan nama kelas untuk inference
with open(CLASSES_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)

pin_mem = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=pin_mem)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin_mem)

print(f'Classes (setelah whitelist): {class_names}')
print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

Classes (setelah whitelist): ['Angular Leafspot', 'Leaf Mildew Powdery', 'Leaf Spot', 'normal']
Train samples: 9522, Val samples: 2380


## 5. Model GoogleNet

In [12]:
def create_googlenet_model(num_classes):
    """Load GoogleNet pretrained dan modifikasi classifier untuk num_classes"""
    model = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
    # GoogleNet memiliki fc layer di model.fc
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = create_googlenet_model(num_classes)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print(model)

GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

## 6. Training Loop

In [13]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = logits.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            loss = criterion(logits, labels)
            total_loss += loss.item()
            _, pred = logits.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

In [14]:
from tqdm.auto import tqdm

best_acc = 0.0
for epoch in tqdm(range(EPOCHS), desc="Training"):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    
    msg = f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}'
    print(msg, flush=True)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'num_classes': num_classes,
            'class_names': class_names,
            'epoch': epoch,
            'val_acc': val_acc
        }, MODEL_PATH)
        print(f'  -> Model saved (acc: {val_acc:.4f})', flush=True)

print(f'\nTraining selesai. Best val accuracy: {best_acc:.4f}', flush=True)

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 0.0743 Acc: 0.9762 | Val Loss: 7.7586 Acc: 0.1534
  -> Model saved (acc: 0.1534)
Epoch 2/5 | Train Loss: 0.0339 Acc: 0.9884 | Val Loss: 10.2980 Acc: 0.1500
Epoch 3/5 | Train Loss: 0.0079 Acc: 0.9976 | Val Loss: 9.0280 Acc: 0.1542
  -> Model saved (acc: 0.1542)
Epoch 4/5 | Train Loss: 0.0276 Acc: 0.9911 | Val Loss: 6.5580 Acc: 0.1429
Epoch 5/5 | Train Loss: 0.0103 Acc: 0.9965 | Val Loss: 8.9463 Acc: 0.1555
  -> Model saved (acc: 0.1555)

Training selesai. Best val accuracy: 0.1555


## 7. Download Model

In [15]:
# Google Colab: Download file
if IN_COLAB:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download(CLASSES_PATH)
    print('Model dan class_names berhasil didownload!')
else:
    # Kaggle: File tersedia di Output tab
    print(f'Model disimpan di: {MODEL_PATH}')
    print(f'Class names: {CLASSES_PATH}')
    print('Di Kaggle: Add Output di kanan, lalu download dari Output tab')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model dan class_names berhasil didownload!


In [16]:
MODEL_PATH

'/content/output/googlenet_model.pth'

In [17]:
print(f'Model disimpan di: {MODEL_PATH}')
print(f'Class names: {CLASSES_PATH}')
print('Di Kaggle: Add Output di kanan, lalu download dari Output tab')

Model disimpan di: /content/output/googlenet_model.pth
Class names: /content/output/class_names.json
Di Kaggle: Add Output di kanan, lalu download dari Output tab


In [20]:
import os
import shutil
from IPython.display import FileLink, display

TARGET_DIR = "/kaggle/working/strawberry-1"

# buat folder jika belum ada
os.makedirs(TARGET_DIR, exist_ok=True)

# path tujuan
model_target = os.path.join(TARGET_DIR, os.path.basename(MODEL_PATH))
classes_target = os.path.join(TARGET_DIR, os.path.basename(CLASSES_PATH))

# pindahkan file
shutil.move(MODEL_PATH, model_target)
shutil.move(CLASSES_PATH, classes_target)

print(f'Model disimpan di: {model_target}')
print(f'Class names: {classes_target}')

# buat link download
display(FileLink(model_target))
display(FileLink(classes_target))

Model disimpan di: /kaggle/working/strawberry-1/googlenet_model.pth
Class names: /kaggle/working/strawberry-1/class_names.json


/kaggle/working/strawberry-1/googlenet_model.pth

/kaggle/working/strawberry-1/class_names.json